<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_12_3_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install neuralforecast -q
!pip install kaleido==0.2.1 -q
!pip install workalendar -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from workalendar.europe import Russia
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
HORIZON_NAME = "24h"

In [ ]:
def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id': house_ids,
        'y_true':y_true,
        'y_pred': y_pred,
        'horizon': horizon_name,
        'model': model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8, "8h": 16, "24h": 48,
    "7d":  336, "14d": 672, "1m": 1488,
}

HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12},
    'house_2': {'n_flats': 191, 'n_floors': 12},
    'house_3': {'n_flats': 124, 'n_floors': 12},
    'house_4': {'n_flats': 263, 'n_floors': 12},
    'house_5': {'n_flats': 127, 'n_floors':  7},
    'house_6': {'n_flats': 497, 'n_floors': 25},
    'house_7': {'n_flats': 471, 'n_floors': 17},
    'house_8': {'n_flats': 171, 'n_floors': 23},
}

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

dfs = []
for house in houses:
    tmp = pd.DataFrame({
        "unique_id": house,
        "ds": df["timestamp"],
        "y": df[house],
    })
    dfs.append(tmp)

df_nf = pd.concat(dfs, ignore_index=True)

df_nf['y'] = df_nf.groupby('unique_id')['y'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

static_df = pd.DataFrame([
    {'unique_id': house, 'n_flats': HOUSE_META[house]['n_flats'],
     'n_floors': HOUSE_META[house]['n_floors']}
    for house in houses
])

df_val = df_nf[
    (df_nf["ds"] >= df["timestamp"].iloc[train_end]) &
    (df_nf["ds"] < df["timestamp"].iloc[val_end])
]
df_test = df_nf[df_nf["ds"] >= df["timestamp"].iloc[val_end]]

val_size = len(df_val) // len(houses)
test_size = len(df_test) // len(houses)

horizon_points = horizons[HORIZON_NAME]

In [ ]:
torch.cuda.empty_cache()
gc.collect()

nf = NeuralForecast(
    models=[
        TFT(
            h=horizon_points,
            input_size=48,
            hidden_size=32,
            n_head=2,
            max_steps=200,
            val_check_steps=50,
            early_stop_patience_steps=5,
            scaler_type="minmax",
            accelerator="gpu",
            devices=1,
        )
    ],
    freq="30min"
)

cv_df = nf.cross_validation(
    df=df_nf,
    static_df=static_df,
    val_size=val_size,
    test_size=test_size,
    n_windows=None,
)

results = {}
for house in houses:
    metrics = evaluate(
        cv_df[cv_df["unique_id"] == house]["y"].values,
        cv_df[cv_df["unique_id"] == house]["TFT"].values
    )
    results[house] = metrics
    print(f"{house}: MAE={metrics['MAE']:.3f}, RMSE={metrics['RMSE']:.3f}, MAPE={metrics['MAPE']:.3f}%")

    # сохраняем только последний тестовый период
    last_cutoff = cv_df[cv_df["unique_id"] == house]["cutoff"].max()
    mask_test = (cv_df["unique_id"] == house) & (cv_df["cutoff"] == last_cutoff)

    save_predictions(
        ts=cv_df.loc[mask_test, "ds"].values,
        house_ids=np.full(mask_test.sum(), house),
        y_true=cv_df.loc[mask_test, "y"].values,
        y_pred=cv_df.loc[mask_test, "TFT"].values,
        horizon_name=HORIZON_NAME,
        model_name="TFT",
        filename="predictions_tft.csv"
    )

rows = [{"модель": "TFT", "дом": h, "горизонт": HORIZON_NAME, **results[h]} for h in houses]
pd.DataFrame(rows).to_csv(f"results_tft_{HORIZON_NAME}.csv", index=False)

# сохраняем cv_df для графика
cv_df_house1 = cv_df[cv_df["unique_id"] == "house_1"]
cv_df_house1.to_csv(f"cv_tft_{HORIZON_NAME}_house1.csv", index=False)


print(f'Размер predictions_tft.csv: '
      f'{os.path.getsize("predictions_tft.csv") / 1024 / 1024:.1f} МБ')

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 2 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 3 │ embedding               │ TFTEmbedding             │    128 │ train │     0 │
│ 4 │ temporal_encoder        │ TemporalCovariateEncoder │ 39.6 K │ train │     0 │
│ 5 │ temporal_fusion_decoder │ TemporalFusionDecoder    │ 17.0 K │ train │     0 │
│ 6 │ output_adapter          │ Linear                   │     33 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 56.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 56.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 88                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=200` reached.


INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

house_1: MAE=5.349, RMSE=7.067, MAPE=10.247%
house_2: MAE=2.229, RMSE=2.920, MAPE=10.135%
house_3: MAE=1.731, RMSE=2.266, MAPE=11.930%
house_4: MAE=3.400, RMSE=4.442, MAPE=10.119%
house_5: MAE=2.149, RMSE=2.860, MAPE=9.422%
house_6: MAE=6.348, RMSE=8.300, MAPE=9.082%
house_7: MAE=7.031, RMSE=9.104, MAPE=8.195%
house_8: MAE=2.343, RMSE=3.046, MAPE=7.527%
Размер predictions_tft.csv: 0.0 МБ


In [ ]:
cv_df_house1 = cv_df[cv_df["unique_id"] == "house_1"]
cv_df_house1.to_csv(f"cv_tft_{HORIZON_NAME}_house1.csv", index=False)